# Noise blend schedules (`main_model.py` formulas)

**Blend:** ε = w·ε_white + (1−w)·ε_blue.

Set **`W_START`** to **`0`**, **`0.25`**, or **`0.5`**:

- **`0.5`** — **sigmoid** schedule (`noise_blend_schedule: sigmoid` in YAML) uses YAML `gamma_start` / `gamma_end` / `gamma_tau`. **Linear** and **step** use **w₀ = 0.5** at t=0. **Step** switch for these plots is **`STEP_SWITCH_AT`** in the first code cell (default **10**: w=w₀ for i<10, w=1 for i≥10).

- **`0`** or **`0.25`** — **same start w₀ for linear and step**; **sigmoid** uses **γ_s = logit(w₀)** so **σ(γ_s)=w₀** at t=0. For **w₀=0**, **γ_s = logit(ε)** with float64 **ε** so σ(γ_s)=ε (no arbitrary constants).

Second cell: **w vs u** for the **sigmoid** curve from your choice.

**If linear/step still start at 0 when you chose 0.5:** restart the kernel and **run all cells in order**, or at least re-run the **setup** cell after changing `W_START` (stale `w_lin_step` causes old plots).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml

REPO = Path.cwd()
cfg_file = REPO / "config" / "base.yaml"
if not cfg_file.is_file():
    raise FileNotFoundError(f"Expected {cfg_file}; run from repo root.")

with open(cfg_file) as f:
    cfg = yaml.safe_load(f)

num_steps = int(cfg["diffusion"]["num_steps"])
m = cfg.get("model", {})
gamma_start_cfg = float(m.get("gamma_start", 0.0))
gamma_end = float(m.get("gamma_end", 3.0))
gamma_tau = float(m.get("gamma_tau", 0.2))
noise_blend_w_start_cfg = float(m.get("noise_blend_w_start", 0.0))
_step_t = m.get("noise_blend_step_t")
if _step_t is None:
    step_from_yaml = max(0, min(num_steps // 2, num_steps))
else:
    step_from_yaml = int(max(0, min(int(_step_t), num_steps)))

# Step plot: w=w0 for diffusion index i < STEP_SWITCH_AT, else 1 (edit to match training --noise-blend-step-t)
STEP_SWITCH_AT = 10
step_blue_steps = max(0, min(int(STEP_SWITCH_AT), num_steps))

print("num_steps:", num_steps, "| YAML γ_s:", gamma_start_cfg, "γ_e:", gamma_end, "τ:", gamma_tau)
print("YAML noise_blend_w_start:", noise_blend_w_start_cfg)
print("step (plots): i <", step_blue_steps, "→ w0;  YAML default would be i <", step_from_yaml)

In [ ]:
def weights_sigmoid(t, gamma_s, ge, gtau, Tn):
    t = np.asarray(t, dtype=np.float64)
    x = gamma_s + (ge - gamma_s) * ((t / Tn) ** gtau)
    x = np.clip(x, -709.0, 709.0)
    return 1.0 / (1.0 + np.exp(-x))


def weights_linear(t, w0, Tn):
    t = np.asarray(t, dtype=np.float64)
    denom = max(Tn - 1, 1)
    u = np.clip(t / denom, 0.0, 1.0)
    return w0 + (1.0 - w0) * u


def weights_step(t, w0, step_bt, Tn):
    idx = np.asarray(t, dtype=np.int64)
    idx = np.clip(idx, 0, Tn - 1)
    return np.where(idx < step_bt, w0, 1.0)


def gamma_start_for_w0(w0: float) -> float:
    """σ(γ_s) = w0; w0≤0 → p=ε; w0≥1 → p=1−ε."""
    eps = np.finfo(np.float64).eps
    if w0 <= 0.0:
        p = eps
    elif w0 >= 1.0:
        p = 1.0 - eps
    else:
        p = float(w0)
    return float(np.log(p / (1.0 - p)))


W_START = 0.5  # 0.0, 0.25, or 0.5 — re-run this cell after editing, then re-run the plot cell

_w = float(W_START)
if not any(abs(_w - x) < 1e-12 for x in (0.0, 0.25, 0.5)):
    raise ValueError("W_START must be 0.0, 0.25, or 0.5")

t_int = np.arange(num_steps, dtype=np.float64)

if abs(_w - 0.5) < 1e-12:
    gamma_s = gamma_start_cfg
    w_lin_step = 0.5  # linear & step start at 0.5 with YAML sigmoid schedule
else:
    gamma_s = gamma_start_for_w0(_w)
    w_lin_step = _w

w_sigmoid = weights_sigmoid(t_int, gamma_s, gamma_end, gamma_tau, num_steps)
w_linear = weights_linear(t_int, w_lin_step, num_steps)
w_step = weights_step(t_int, w_lin_step, step_blue_steps, num_steps)

print(f"W_START={_w:g}  →  γ_s={gamma_s:.6g}  linear/step w₀={w_lin_step:g}")
print(f"sanity: w_sigmoid[0]={w_sigmoid[0]:.6g}  w_linear[0]={w_linear[0]:.6g}  w_step[0]={w_step[0]:.6g}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_int, w_sigmoid, "-", label="sigmoid", lw=2)
ax.plot(t_int, w_linear, "--", label="linear", lw=2)
ax.plot(t_int, w_step, "-.", label="step", lw=2)
ax.set_xlabel("t")
ax.set_ylabel("w")
ax.set_title(f"w₀ = {_w:g}")
ax.set_xlim(-0.5, num_steps - 0.5)
ax.set_ylim(-0.02, 1.05)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
u_warp = (t_int / num_steps) ** gamma_tau
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(u_warp, w_sigmoid, "-", lw=2, color="C0")
ax.set_xlabel("u")
ax.set_ylabel("w")
ax.set_title("w vs u")
plt.tight_layout()
plt.show()

### Optional: `ipywidgets` dropdown (0 / 0.25 / 0.5)

Same rules as **`W_START`** above.

In [ ]:
try:
    import ipywidgets as widgets

    def _gamma_start_for_w0(w0: float) -> float:
        eps = np.finfo(np.float64).eps
        if w0 <= 0.0:
            p = eps
        elif w0 >= 1.0:
            p = 1.0 - eps
        else:
            p = float(w0)
        return float(np.log(p / (1.0 - p)))

    def plot_for_w(w):
        w = float(w)
        if w == 0.5:
            gs = gamma_start_cfg
            wls = 0.5
        else:
            gs = _gamma_start_for_w0(w)
            wls = w
        g = weights_sigmoid(t_int, gs, gamma_end, gamma_tau, num_steps)
        lin = weights_linear(t_int, wls, num_steps)
        st = weights_step(t_int, wls, step_blue_steps, num_steps)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(t_int, g, "-", label="sigmoid", lw=2)
        ax.plot(t_int, lin, "--", label="linear", lw=2)
        ax.plot(t_int, st, "-.", label="step", lw=2)
        ax.set_xlabel("t")
        ax.set_ylabel("w")
        ax.set_title(f"w₀ = {w:g}")
        ax.set_xlim(-0.5, num_steps - 0.5)
        ax.set_ylim(-0.02, 1.05)
        ax.legend(loc="lower right")
        plt.tight_layout()
        plt.show()

    widgets.interact(
        plot_for_w,
        w=widgets.Dropdown(options=[0.0, 0.25, 0.5], value=0.25, description="w₀"),
    )
except ImportError:
    print("ipywidgets not installed; set W_START in the setup cell.")